# Fine-tune RoBERTa classifiers (speaker + pair)

**Runtime → Change runtime type → T4 GPU.** Everything reads from `data/classifiers/`, already committed, so no data rebuild is needed.

TF-IDF baselines to beat (episode-held-out test set):

| Task | Acc | Macro-F1 | Chance |
|---|---|---|---|
| Speaker, 6-way, 1 utterance | 0.291 | 0.289 | 0.167 |
| Pair, 15-way, 1 segment | 0.208 | 0.172 | 0.067 |

**Before running, write down your predictions:**
1. Which task gains more from RoBERTa over TF-IDF, and why? (Think about data size per class and what each model can represent.)
2. The speaker run trains for 3 epochs on ~35k very short texts. Will val F1 peak at epoch 1, 2 or 3?
3. For the pair task, which pairs do you expect the model to confuse most? Check against the confusion matrix in `results/classifiers/pair_tfidf_confusion.png` afterwards.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

If the repo is private, add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) holding a fine-grained token with read access to the repo.

In [ ]:
REPO = "YOUR_GITHUB_USERNAME/329x-friends"  # <- edit

import os
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("329x-friends"):
    !git clone -q {url} 329x-friends
%cd 329x-friends
!git log --oneline -3

In [ ]:
!pip -q install -U transformers accelerate spacy
!python -m spacy download en_core_web_sm -q

## 1. Speaker classifier
About 1.1k steps/epoch at `max_len=64`, roughly 3–4 min/epoch on a T4. The script carves an episode-grouped 10% validation set out of train, keeps the best epoch by val macro-F1, and touches test once.

In [ ]:
!python scripts/train_transformer.py --task speaker --model roberta-base --epochs 3 --batch-size 32 --lr 2e-5

## 2. Pair classifier
Segments are joined into an `S1:`/`S2:` transcript; 512 tokens covers ~99.5% of them. With only ~2.5k segments, use more epochs and a smaller batch.

In [ ]:
!python scripts/train_transformer.py --task pair --model roberta-base --epochs 6 --batch-size 16 --lr 2e-5 --max-len 512

## 3. Compare against TF-IDF

In [ ]:
import json, glob, pandas as pd
rows = []
for p in sorted(glob.glob("results/classifiers/*metrics.json")):
    m = json.load(open(p))
    rows.append({"file": os.path.basename(p), "task": m["task"], "model": m["model"],
                 "test_acc": round(m["test_accuracy"], 3), "test_macro_f1": round(m["test_macro_f1"], 3),
                 "chance": round(m["chance_uniform"], 3)})
pd.DataFrame(rows)

In [ ]:
# per-class F1: where does each model win?
def per_class(path):
    pc = json.load(open(path))["per_class"]
    return {k: round(v["f1-score"], 3) for k, v in pc.items() if isinstance(v, dict) and k not in ("macro avg", "weighted avg")}
for task in ["speaker", "pair"]:
    print(task)
    display(pd.DataFrame({"tfidf": per_class(f"results/classifiers/{task}_tfidf_metrics.json"),
                          "roberta": per_class(f"results/classifiers/{task}_roberta-base_metrics.json")}))

## 4. Score the Mistral baseline with the fine-tuned judges

In [ ]:
!python scripts/score_generations.py --speaker-model models/speaker_roberta-base --pair-model models/pair_roberta-base \
    --out results/classifiers/baseline_generation_scores_roberta.json

## 5. Save checkpoints and results
`models/` is gitignored (~500MB per checkpoint), so checkpoints go to Drive. Metrics JSONs are small; download them or push them back.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/329x-friends-models
!cp -r models/speaker_roberta-base models/pair_roberta-base /content/drive/MyDrive/329x-friends-models/
!ls /content/drive/MyDrive/329x-friends-models

In [ ]:
# optional: push metrics back (needs a token with write access)
# !git config user.email "you@example.com" && git config user.name "Your Name"
# !git add results/classifiers/*.json && git commit -m "RoBERTa classifier results" && git push